# CopyLane 판정 인코더 — E1-02 OpenAI 임베딩 비교 : 범용 임베딩 + 선형 분류기 대 파인튜닝 인코더 (박수진, 2026-10-14)

> 골든 **`f6ccc5a7de44`** (4차 데이터) · 공통 dev 749행 (지문 `4ebfcb231da703b5`) · 검수판 권소라 10-11 (`f3bfec90…`)
> 기준점 — **D4-10** = KcBERT 파인튜닝(배포 판 후보) · 같은 데이터 · 같은 채점
> 목적 — 발표용 비교. 「GPT(OpenAI) 임베딩을 쓰면 되지 않나」에 답할 수. **채택 후보가 아니다.** 등록 `docs/psj/reports/D4_실험기록_20261012.md` E1-02 (돌리기 전에 올렸다) · 🔴 **dev 만 본다. test 채점 셀이 없다** (D-175)

## 무엇을 하나

데이터 셀(1 ~ 4)은 D4-10 · E1-00 과 **글자 하나 다르지 않다** — 학습 4,511행 · dev 749행 · 정상 문구 측정 행이 같다.
모델 셀만 바꾼다.

| | D4-10 | E1-02 |
|---|---|---|
| 문장 → 벡터 | KcBERT 를 함께 학습(파인튜닝) | OpenAI `text-embedding-3-small` · `text-embedding-3-large` (고정 · 학습 안 함) |
| 분류기 | KcBERT 위 선형 층 · Focal 손실 | 유형마다 로지스틱 회귀 (표준화 · C 1.0 · class_weight balanced · 고정값 · dev 로 고르지 않음) |
| 품목 입력 | `[CLS] 품목 [SEP] 문장` · 학습 때 50% 「모름」 | `품목: 화장품\n문장: …` · 품목 있는 학습 행은 품목판 0.5 + 「모름」판 0.5 가중치로 두 번 |
| 시드 | 48 · 49 · 50 | 결정적이라 한 번 학습 · 채점 스크립트에 맞춰 같은 확률을 시드 셋 이름으로 적는다 |

## 돌리는 법

1. Colab 왼쪽 🔑(보안 비밀)에 `OPENAI_API_KEY` 를 넣고 이 노트북에 접근을 허용한다. **키를 셀에 적지 않는다.**
2. 런타임은 CPU 로 충분하다. 위에서부터 차례로 실행 · 업로드 셀에서 `golden.jsonl` 과 검수판 CSV 를 올린다.
3. 임베딩은 Drive `copylane_E1-02/emb_*.npz` 에 저장해 두어 다시 돌려도 API 를 다시 부르지 않는다. 문장 약 1.1만 개 · 토큰 약 50만 개 (실행 전에 가격을 본다).
4. 맨 아래 셀이 CSV 24개(모델 2 × 입력 2 × 시드 3 × dev · 정상 문구)를 **`E1-02_csv.zip`** 으로 묶는다. PC 의 `Downloads\copylane_D4\E1-02_csv\` 에 푼다.

🔴 골든 문장이 OpenAI API 로 나간다 — 공개 출처(공정위 · 식약처) 문장과 합성 정상 문구다. 행 id 는 보내지 않는다.


## 1. 환경 설정 + 파일 업로드 (골든 · 정상 합성 문구 CSV)

`sympy` 충돌로 `from transformers import ...` 가 깨지면 런타임 > 세션 다시 시작 후 첫 셀은 건너뛰고 다음 셀부터 실행한다.

In [ ]:
!pip install -q -U sympy transformers accelerate datasets scikit-learn pandas openai

In [ ]:
import os, json, hashlib, shutil, gc, math, random, csv, time, zipfile
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import files as colab_files          # Colab — 없으면 묻는다
except ImportError:
    colab_files = None                                     # RunPod · 로컬 — 두 파일을 이 노트북 옆에 미리 둔다
if not os.path.exists("golden.jsonl"):
    if colab_files is None:
        raise SystemExit("🔴 golden.jsonl 이 없다 — 이 노트북과 같은 폴더에 둔다 (RunPod: VS Code 탐색기로 끌어 놓기)")
    print("① golden.jsonl 선택")
    colab_files.upload()
SYN_NAME = "LLM_카피생성_정상카피_900_판독용_사람칸채움_ksr_2026-10-11.csv"     # 🆕 카드 17 — 권소라 검수판 (판독(사람) 칸)
if not os.path.exists(SYN_NAME):
    if colab_files is None:
        raise SystemExit(f"🔴 {SYN_NAME} 이 없다 — 이 노트북과 같은 폴더에 둔다")
    print("② 정상 합성 문구 CSV 선택 —", SYN_NAME)
    for _f in colab_files.upload():                  # 브라우저가 이름을 바꿔 올렸으면(「 (1)」 등) 제 이름으로 둔다
        if _f.endswith(".csv") and _f != SYN_NAME:
            os.replace(_f, SYN_NAME)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

GOLDEN_SHA = sha256("golden.jsonl")
print("golden.jsonl sha256:", GOLDEN_SHA[:16], "…")
# D-176: 입력 지문을 **대조**한다
EXPECTED_GOLDEN_SHA = "f6ccc5a7de44c6863ebb901d4e5a726acb35c92527e41fd1c3850490158fe744"   # 🆕 10-10 2 차 재동결 (8,383행 · `전제` 칸)
EXPECTED_ROWS = (8383, 5894, 2489)   # 전체 · 학습 · 평가
if GOLDEN_SHA != EXPECTED_GOLDEN_SHA:
    raise SystemExit(f"🔴 golden sha 가 원장과 다르다 — 받은 {GOLDEN_SHA[:16]} · 원장 {EXPECTED_GOLDEN_SHA[:16]} (D-176). "
                     "런타임에 예전 파일이 남았으면 런타임을 새로 시작한다")
SYN_SHA = sha256(SYN_NAME)
EXPECTED_SYN_SHA = "f3bfec9079e05c367b2d27495c385659118fd56abd963e89ac5d760abfdd00ea"      # 🆕 권소라 10-11 검수판 (900행 · 판독(사람) 칸)
if SYN_SHA != EXPECTED_SYN_SHA:
    raise SystemExit(f"🔴 정상 합성 문구 CSV 가 카드에 적은 판과 다르다 — 받은 {SYN_SHA[:16]} · 기대 {EXPECTED_SYN_SHA[:16]}. "
                     "판독 칸을 채운 새 판이면 새 카드로 돌린다")
print("정상 합성 문구 CSV sha256:", SYN_SHA[:16], "…")

## 2. 스위치 — 바꿀 것은 이 셀뿐이다

| 스위치 | 기본값 | 뜻 |
|---|---|---|
| `SAVE_MODEL_ZIP` | `False` | 탐색이라 기본은 저장하지 않는다. 통과하고 팀장이 열자고 하면 `True` 로 다시 돌린다 |
| `SMOKE` | `False` | `True` = 1분짜리 점검(시드 1 · 1 epoch · 학습 300줄). 수치는 의미 없다 |
| `CAT_DROP` · `MAX_LEN` | 0.5 · 136 | 🔒 **D4 기록에 적은 값이다. 고치지 않는다** |
| `SYN_HOLD_PER_CAT` · `SYN_SEED` | 5 · 42 | 🔒 정상 문구를 나누는 방법 — 카드 17 과 같다 |


In [ ]:
EMB_MODELS = {"emb3-small": "text-embedding-3-small", "emb3-large": "text-embedding-3-large"}   # 🆕 E1-02 — 판 이름 → OpenAI 모델
RUN = "E1-02"
SAVE_MODEL_ZIP = False             # E1-02 — 저장할 모델이 없다
SMOKE = False

# ── 🔒 입력 — D4-04 와 같다 (D-336)
CAT_VALUES = ["식품", "건기식", "화장품"]     # 골든 `품목` 칸 · 정상 문구 CSV `품목` 칸의 값. 그 밖 · 빈 칸은 「모름」
CAT_UNKNOWN = "모름"
CAT_DROP = 0.50                              # 학습 때 품목이 있는 행을 「모름」으로 바꿔 넣는 비율 (배치마다 새로 뽑는다 · 옛 카드 6 은 0.30)
MAX_LEN = 136                                # [CLS] 품목 [SEP] 문장 [SEP] 전체 — D4-00 은 문장만 128
MODES = ["품목", "모름"]                     # (가) 품목을 넣는다 / (나) 전부 「모름」

# ── 학습 설정 — D4-00 · D4-04 와 같다. 바꾸지 않는다
MODEL_NAME = None                  # E1-02 — 파인튜닝 백본 없음 (임베딩은 EMB_MODELS)
LR_C, LR_CLASS_WEIGHT, LR_MAX_ITER = 1.0, "balanced", 3000   # 🔒 분류기 — 고정값 · dev 로 고르지 않는다
SEEDS = [48, 49, 50]                # D4-00 과 같은 시드 — 짝짓는다
MAX_EPOCHS, PATIENCE = 6, 2
BATCH, LR, WARMUP = 32, 2e-5, 0.1
WEIGHT_DECAY = 0.01
FOCAL_GAMMA, POS_WEIGHT_CAP = 1.5, 5
USE_WORD_ROWS = False             # 사례집 낱말 행 제외
DEV_FRAC, DEV_SEED = 0.15, 42     # 공통 dev — 바꾸면 dev 행이 달라진다
TH_OFF = 1.01                     # 편입 대기 칸(기능성화장품_오인)의 문턱 — 꺼 둠 (D-321)
SCORE_VIEW = "v11"                # 회차 · 시드는 공통 dev 749행 전부로 고른다 — 합성 정상 dev 로는 아무것도 고르지 않는다

# ── 🔒 정상 합성 문구를 나누는 방법 — 카드 17 과 같다
SYN_HOLD_PER_CAT, SYN_SEED = 5, 42
SYN_REVIEW_OK = "적법"            # 🔒 D-325
SYN_FACT_OK = "적법(사실 전제)"   # 🆕 D4-10 — 학습 음성으로 더하는 검수 낱말 (D-331 결정 7)
SYN_EXPECT = {"학습": 334, "학습-사실 전제": 211, "합성 정상 dev": 171, "참고(사실 전제 dev · 애매 · 조건부)": 184}      # 로컬 대조 10-13
QUIET_MARGIN = 0.5                # 판정 로직의 여유 구간 (미리 보기용)

assert 0 <= CAT_DROP < 1 and CAT_UNKNOWN not in CAT_VALUES
if SMOKE:
    SEEDS, MAX_EPOCHS = [48], 1
NOTEBOOK = "E1-02"
def run_tag(name):
    return f"copylane-openai-E1-02-{name}"

OUT_ROOT = "/content/drive/MyDrive/copylane_E1-02" + ("_smoke" if SMOKE else "")
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(OUT_ROOT, exist_ok=True)
except Exception as e:
    OUT_ROOT = "./copylane_E1-02" + ("_smoke" if SMOKE else "")      # RunPod · 로컬 — 노트북 옆 폴더
    os.makedirs(OUT_ROOT, exist_ok=True)
    print("ℹ️ Colab 이 아니다 — 결과를", os.path.abspath(OUT_ROOT), "에 둔다 (파드를 Terminate 하면 사라진다 · 끝나면 E1-02_csv.zip 을 받는다)")
print("실험:", RUN, "·", EMB_MODELS, "· 결과 폴더", OUT_ROOT)


## 3. 데이터 + dev 분리 — 공통 dev (Baseline 노트북과 같은 셀)

1. 묶음 = 같은 원문에서 나온 행 (`inj:T1:hf:0:0` → `hf:0:0` · 그 밖은 `id` 의 `#` 앞).
2. 원천마다 묶음의 15% 를 dev 로 (시드 42 · 묶음 이름순 → 섞기).
3. 6종마다 dev 양성이 10건보다 적으면 남은 묶음에서 채운다.
4. dev 행 = dev 묶음의 **문장 단위 · 합성 아님** 행.

같은 골든이면 **749줄 · 지문 `4ebfcb231da703b5`** 가 나와야 한다 — v11 의 `dev_ids_v11_seed42.csv` 와 같은 행 · 같은 순서다. 다르면 멈춘다.

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

SEL = ["질병_예방치료_표방", "건강기능식품_오인", "의약품_오인", "거짓_과장", "소비자_기만", "후기_체험기_기만"]   # 모델을 고르는 6종 (D-321 결정 4)
NEW_CONF = ["부당_비교광고", "비방광고"]
CONF8 = SEL + NEW_CONF                       # 확정 8종 — 문턱을 dev 에서 고른다
WAITING = ["기능성화장품_오인"]              # 편입 대기 — 칸은 두고 문턱은 끈다
LABEL_LIST = CONF8 + WAITING                 # 저장 순서 (팀장 10-06)
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
N_SEL, N8, N_ALL = len(SEL), len(CONF8), len(LABEL_LIST)

def norm_text(s):
    return " ".join(s.split())

def group_key(rid):
    if rid.startswith("inj:"):               # 'inj:T1:hf:0:0' → 원본 승인 문구 'hf:0:0'
        return rid.split(":", 2)[2]
    return rid.split("#")[0]

def encoder_target(r):
    """인코더의 표적 — `docs/psj/e2e_prototype/score_encoder_dev.py` 의 `encoder_target` 과 같다 (🆕 카드 17).
    조건 M · D · L 은 음성 (D-296 개정 2) · 전제 `건기식_비인정` 행은 `거짓_과장` → `건강기능식품_오인` (D-326 보충 12) ·
    근거에 4 호 나목 말고 다른 4 호 목이 있으면 `거짓_과장` 도 남긴다. 🚨 골든 labels 는 고치지 않는다."""
    if r.get("조건") in ("M", "D", "L"):
        return []
    t = list(r.get("labels") or [])
    if r.get("전제") == "건기식_비인정" and "거짓_과장" in t:
        other4 = any("제4호" in g and "|나목" not in g for g in (r.get("근거") or []))
        t = [x for x in t if x != "거짓_과장" or other4]
        if "건강기능식품_오인" not in t:
            t.append("건강기능식품_오인")
    return t

def normalize(r):
    cond, labels = r.get("조건"), list(r.get("labels") or [])
    return {"id": r["id"], "group": group_key(r["id"]), "text": r["text"], "labels": labels,
            # target 이 학습 · 채점의 정답이다 — 조건 M · D · L 음성 · 전제 건기식_비인정 표적 변경 (encoder_target)
            "target": encoder_target(r), "premise": r.get("전제"),
            "split": r["split"], "provenance": r.get("provenance"), "origin": r.get("origin"),
            "zone": r.get("구역"), "cond": cond, "unit": r.get("unit"),
            "cat": r.get("품목")}     # 🆕 D4-04 — 품목 칸을 읽는다

def is_reason(r):
    return r["provenance"] == "ftc_decisions_body" and r["zone"] == "이유"

def is_inj(r):
    return r["origin"] == "injected" or r["id"].startswith("inj:")

def bucket(r):
    # 8종 기준 (run_judge_dist.py 와 같은 갈래)
    if r["cond"] == "M": return "M"
    if r["cond"] == "D": return "D"
    if r["cond"] == "L": return "L"
    if any(l in CONF8 for l in r["labels"]): return "scored"
    if r["labels"]: return "대기"
    return "pending" if r["cond"] in ("C", "A", "B") else "neg"

golden = load_jsonl("golden.jsonl")
all_rows = [normalize(r) for r in golden]
unknown = {l for r in all_rows for l in r["labels"]} - set(LABEL_LIST)
assert not unknown, f"🔴 예상 밖 라벨: {unknown}"
train_all = [r for r in all_rows if r["split"] == "train"]
_got = (len(all_rows), len(train_all), len(all_rows) - len(train_all))
if _got != EXPECTED_ROWS:
    raise SystemExit(f"🔴 행 수가 다르다 — 받은 {_got} · 기대 {EXPECTED_ROWS} (D-176)")
del golden, all_rows      # 🔴 여기서부터 test 행은 메모리에도 없다 — 이 노트북은 train(→ train · dev)만 본다

# ══ dev 분리 — v11 (encoder_finetune_colab_v11_kcbert_1006) 섹션 3 과 같은 코드
def dev_eligible(r):
    return r["unit"] == "문장" and r["origin"] != "injected"

groups = defaultdict(list)
for r in train_all:
    groups[r["group"]].append(r)

prov_groups = defaultdict(list)             # 원천별 15% (묶음 단위 · 시드 고정)
for g in sorted(groups):
    elig = [r for r in groups[g] if dev_eligible(r)]
    if elig:
        prov_groups[Counter(r["provenance"] for r in elig).most_common(1)[0][0]].append(g)
rng = random.Random(DEV_SEED)
dev_groups = set()
for prov in sorted(prov_groups):
    gs = prov_groups[prov][:]
    rng.shuffle(gs)
    dev_groups.update(gs[:max(1, math.ceil(DEV_FRAC * len(gs)))])

def conf_pos(rows, l):
    return sum(1 for r in rows if dev_eligible(r) and l in r["target"])
rest = [g for g in sorted(groups) if g not in dev_groups]
rng.shuffle(rest)
for l in SEL:                               # 6종마다 dev 양성이 10건보다 적으면 채운다
    total = conf_pos(train_all, l)
    want = min(10, math.ceil(DEV_FRAC * total))
    have = sum(conf_pos(groups[g], l) for g in dev_groups)
    for g in rest:
        if have >= want:
            break
        k = conf_pos(groups[g], l)
        if k and g not in dev_groups:
            dev_groups.add(g); have += k

dev_rows = [r for g in sorted(dev_groups) for r in groups[g] if dev_eligible(r)]
dev_texts = {norm_text(r["text"]) for r in dev_rows}
DEV_SHA = hashlib.sha256("|".join(sorted(r["id"] for r in dev_rows)).encode()).hexdigest()[:16]
DEV_EXPECT = {"529c970556d77bd2c231526e20531294bf469f794e74a4796835a36f456eb7fe": (749, "4ebfcb231da703b5"),
              "f6ccc5a7de44c6863ebb901d4e5a726acb35c92527e41fd1c3850490158fe744": (749, "4ebfcb231da703b5")}   # 🆕 10-10 판 — dev 행은 같다   # 골든 sha → v11 dev (줄 수 · 지문)
if GOLDEN_SHA in DEV_EXPECT:
    if (len(dev_rows), DEV_SHA) != DEV_EXPECT[GOLDEN_SHA]:
        raise SystemExit(f"🔴 dev 가 v11 과 다르다 — {len(dev_rows)}줄 · 지문 {DEV_SHA} · 기대 {DEV_EXPECT[GOLDEN_SHA]}. 분리 코드를 건드렸는지 본다")
    print(f"✅ dev = v11 과 같은 행: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
else:
    print(f"🟡 이 골든의 v11 dev 지문을 모른다 — 대조 없이 간다: 묶음 {len(dev_groups)}개 · {len(dev_rows)}줄 · 지문 {DEV_SHA}")
assert len({r["id"] for r in dev_rows}) == len(dev_rows), "dev 에 같은 id 가 둘 이상이다"

# ── dev 행의 갈래 · 채점 방식
for r in dev_rows:
    r["bucket"] = bucket(r)
B = np.array([r["bucket"] for r in dev_rows])
IS_REASON = np.array([is_reason(r) for r in dev_rows])
VIEW = {"v11": np.ones(len(dev_rows), dtype=bool),
        "A": ~IS_REASON & ~np.isin(B, ["M", "pending"])}
SEL_MASK = VIEW[SCORE_VIEW]
BUCKETS = ["scored", "대기", "pending", "M", "D", "L", "neg"]
BUCKET_KO = {"scored": "유형 있는 위반", "대기": "편입 대기 유형만", "pending": "유형 없는 위반", "M": "조건 M", "D": "조건 D", "L": "조건 L(적법)", "neg": "그 밖 음성"}
print("\ndev 갈래 (전체 / 그중 「이유」 구역):")
for b in BUCKETS:
    if (B == b).any():
        print(f"  {BUCKET_KO[b]:12s} {int((B == b).sum()):4d} / {int(((B == b) & IS_REASON).sum()):4d}")
print(f"  합계           {len(dev_rows):4d} / {int(IS_REASON.sum()):4d}")
print(f"채점 행 — v11 방식 {int(VIEW['v11'].sum())}줄 · A 방식 {int(VIEW['A'].sum())}줄  → 이번 선택 방식: {SCORE_VIEW}")
_moved = [r for r in train_all if r.get("premise") == "건기식_비인정" and r["target"] != ([] if r["cond"] in ("M", "D", "L") else r["labels"])]
print(f"표적을 바꾼 행 (전제 건기식_비인정 · D-326 보충 12): {len(_moved)} — 그중 dev {sum(r['id'] in {d['id'] for d in dev_rows} for r in _moved)}")
if GOLDEN_SHA == EXPECTED_GOLDEN_SHA and not SMOKE:
    assert len(_moved) == 37 and sum(r["id"] in {d["id"] for d in dev_rows} for r in _moved) == 7, "표적 변경 행 수가 팀장 전달(학습 30 · dev 7)과 다르다"

## 4. 학습셋 · 품목 · 정상 합성 문구 나누기 (🆕 사실 전제 211 을 학습에)

골든에서 오는 학습 행을 고르는 규칙은 D4-00 과 같다 — dev 묶음을 뺀 train 에서 dev 와 같은 문구 제거 → 조건 M 제외 → 유형 없는 위반 제외 → 낱말 행 제외.
**다른 곳 하나** — 같은 문구를 합칠 때 **품목이 다르면 따로 둔다**(D4-00 은 품목을 보지 않았다).
정상 문구는 카드 17 과 같은 15개 제품을 뗀다 — 학습: 적법 334 + 사실 전제 211 / 측정: 적법 171 · 사실 전제 110 / 참고: 애매 · 조건부 74. 정상 문구의 품목은 그 CSV 의 `품목` 칸이다.


In [ ]:
base_pool = [r for r in train_all if r["group"] not in dev_groups]
STEPS = [("dev 묶음 뺀 train", len(base_pool))]
pool0 = [r for r in base_pool if norm_text(r["text"]) not in dev_texts];                 STEPS.append(("dev 와 같은 문구 제거", len(pool0)))
pool0 = [r for r in pool0 if r["cond"] != "M"];                                          STEPS.append(("조건 M 제외", len(pool0)))
pool0 = [r for r in pool0 if not (r["cond"] in ("C", "A", "B") and not r["labels"])];    STEPS.append(("유형 없는 위반 제외", len(pool0)))
if not USE_WORD_ROWS:
    pool0 = [r for r in pool0 if r["unit"] != "낱말"];                                   STEPS.append(("낱말 행 제외", len(pool0)))
print(" → ".join(f"{k} {n:,}" for k, n in STEPS))
assert not any(r["cond"] == "M" for r in pool0) and not any(r["target"] for r in pool0 if r["cond"] in ("D", "L"))

def cat_of(r):
    return r["cat"] if r.get("cat") in CAT_VALUES else CAT_UNKNOWN

merged = {}
for r in pool0:                                      # 같은 문구 병합 — 라벨 합집합 · 이유/비이유는 따로 · 🆕 D4-04 품목이 다르면 따로
    k = (cat_of(r), norm_text(r["text"]), is_reason(r))
    if k in merged:
        merged[k]["target"] = sorted(set(merged[k]["target"]) | set(r["target"]))
    else:
        merged[k] = dict(r, target=list(r["target"]))
BASE_TRAIN = list(merged.values())

# ── 정상 합성 문구 — 제품 단위로 나눈다
with open(SYN_NAME, encoding="utf-8-sig", newline="") as f:
    _raw = list(csv.DictReader(f))
assert len(_raw) == 900 and len({r["id"] for r in _raw}) == 900, "정상 합성 문구가 900행이 아니다"
SYN_CATS = ["식품", "건기식", "화장품"]
def syn_row(r):
    return {"id": "syn:" + r["id"], "group": f"syn:{r['품목']}:{r['제품종류']}", "text": r["text"], "labels": [], "target": [],
            "split": "extra", "provenance": "llm_copy_ksr_20261011_reviewed", "origin": "synthetic_lawful", "zone": None, "cond": "정상합성", "unit": "문장",
            "cat": r["품목"], "product": r["제품종류"], "angle": r["각도"], "style": r["문체"],
            # 🆕 카드 17 — 학습 · 목표에는 검수 「적법」 · 애매 표시 없음만. 나머지(애매 · 적법(사실 전제) · 조건부)는 참고로만 잰다 (D-325)
            "review": r["판독(사람)"].strip(), "amb_mark": bool(r["애매표시"].strip()),
            "amb": bool(r["애매표시"].strip()) or r["판독(사람)"].strip() != SYN_REVIEW_OK}
SYN = [syn_row(r) for r in _raw]
assert {r["cat"] for r in SYN} == set(SYN_CATS)
SYN_HOLD = set()
for c in SYN_CATS:
    prods = sorted({r["product"] for r in SYN if r["cat"] == c})
    SYN_HOLD |= {(c, p) for p in random.Random(SYN_SEED).sample(prods, SYN_HOLD_PER_CAT)}
SYN_TRAIN = [r for r in SYN if not r["amb"] and (r["cat"], r["product"]) not in SYN_HOLD]
# 🆕 D4-10 — 「적법(사실 전제)」 · 애매 표시 없음 · 학습 제품 → 학습 음성. 측정 제품의 사실 전제는 갈래 「참고-적법(사실 전제)」로 잰다
SYN_TRAIN_FACT = [r for r in SYN if r["review"] == SYN_FACT_OK and not r["amb_mark"] and (r["cat"], r["product"]) not in SYN_HOLD]
_fact_ids = {r["id"] for r in SYN_TRAIN_FACT}
SYN_AMB = [r for r in SYN if r["amb"] and r["id"] not in _fact_ids]
SYN_DEV = [r for r in SYN if not r["amb"] and (r["cat"], r["product"]) in SYN_HOLD]
SYN_EVAL = SYN_DEV + SYN_AMB                          # 학습에 넣지 않는 행 — 시드마다 확률을 적어 둔다
_got = {"학습": len(SYN_TRAIN), "학습-사실 전제": len(SYN_TRAIN_FACT), "합성 정상 dev": len(SYN_DEV), "참고(사실 전제 dev · 애매 · 조건부)": len(SYN_AMB)}
if _got != SYN_EXPECT:
    raise SystemExit(f"🔴 정상 합성 문구를 나눈 수가 카드와 다르다 — 받은 {_got} · 기대 {SYN_EXPECT}")
_base_texts = {norm_text(r["text"]) for r in BASE_TRAIN}
_dup = [r["id"] for r in SYN if norm_text(r["text"]) in dev_texts or norm_text(r["text"]) in _base_texts]
if _dup:
    raise SystemExit(f"🔴 정상 합성 문구 {len(_dup)}행이 골든의 학습 행 · 공통 dev 와 같은 문구다 (예: {_dup[:3]}) — 음성으로 넣으면 라벨이 부딪친다")
assert not ({r["group"] for r in SYN_TRAIN} & {r["group"] for r in SYN_EVAL if not r["amb"]}), "같은 제품이 학습과 합성 정상 dev 에 함께 있다"
assert not ({norm_text(r["text"]) for r in SYN_TRAIN} & {norm_text(r["text"]) for r in SYN_EVAL}), "학습과 측정에 같은 문구가 있다"
assert not ({r["group"] for r in SYN_TRAIN_FACT} & {r["group"] for r in SYN_EVAL if (r["cat"], r["product"]) in SYN_HOLD}), "같은 제품이 사실 전제 학습과 측정에 함께 있다"
assert not ({norm_text(r["text"]) for r in SYN_TRAIN_FACT} & ({norm_text(r["text"]) for r in SYN_EVAL} | dev_texts)), "사실 전제 학습과 측정 · dev 에 같은 문구가 있다"
print("\n■ 정상 합성 문구 나누기 (품목별)")
display(pd.DataFrame({k: {c: sum(r["cat"] == c for r in rs) for c in SYN_CATS} | {"합계": len(rs)}
                      for k, rs in (("학습", SYN_TRAIN), ("학습-사실 전제", SYN_TRAIN_FACT), ("합성 정상 dev", SYN_DEV), ("참고(사실 전제 dev · 애매 · 조건부)", SYN_AMB))}).T)
print("검수 낱말 (판독(사람)):", dict(Counter(r["review"] for r in SYN)), "— 학습은", repr(SYN_REVIEW_OK), "·", repr(SYN_FACT_OK), "· 목표(측정용 171)는", repr(SYN_REVIEW_OK), "만")
assert all(r["review"] == SYN_REVIEW_OK and not r["amb_mark"] for r in SYN_TRAIN + SYN_DEV), "검수 「적법」 아닌 행이 학습 · 목표에 있다 (D-325)"
assert all(r["review"] == SYN_FACT_OK and not r["amb_mark"] for r in SYN_TRAIN_FACT), "사실 전제 학습에 다른 낱말 · 애매 표시 행이 있다"
print("측정용으로 뗀 제품:", {c: sorted(p for cc, p in SYN_HOLD if cc == c) for c in SYN_CATS})

rows = list(BASE_TRAIN) + list(SYN_TRAIN) + list(SYN_TRAIN_FACT)      # 🆕 D4-10 — 사실 전제 211 을 더한다
if SMOKE:
    rows = list(BASE_TRAIN)
    random.Random(0).shuffle(rows)
    rows = rows[:300] + SYN_TRAIN[:60] + SYN_TRAIN_FACT[:40]
TRAIN_ROWS = rows
_base_n = len({(norm_text(r["text"]), is_reason(r)) for r in pool0})      # D4-00 의 병합 규칙이면 몇 행인가
TRAIN_STATS = {"학습 행": len(rows), "골든 행": len(BASE_TRAIN), "D4-00 병합이면 골든 행": _base_n,
               "정상 합성(음성)": sum(r["origin"] == "synthetic_lawful" for r in rows), "그중 사실 전제": sum(r["review"] == SYN_FACT_OK for r in rows if r["origin"] == "synthetic_lawful"), "음성": sum(1 for r in rows if not r["target"]),
               "품목별": {c: sum(1 for r in rows if cat_of(r) == c) for c in CAT_VALUES + [CAT_UNKNOWN]},
               **{l: sum(l in r["target"] for r in rows) for l in LABEL_LIST}}
print("\n■ 학습셋" + ("   (SMOKE — 줄였다)" if SMOKE else ""))
display(pd.Series({k: v for k, v in TRAIN_STATS.items() if k != "품목별"}).to_frame("값").T)
print("품목별 학습 행:", TRAIN_STATS["품목별"])
print("dev 품목별:", dict(Counter(cat_of(r) for r in dev_rows)), "· 그중 유형 있는 위반:", dict(Counter(cat_of(r) for r in dev_rows if r["bucket"] == "scored")))
EXPECT_TRAIN = {"학습 행": 4511, "골든 행": 3966, "D4-00 병합이면 골든 행": 3964, "정상 합성(음성)": 545, "그중 사실 전제": 211,
                "품목별": {"식품": 1113, "건기식": 1380, "화장품": 400, "모름": 1618}}       # D4-04 의 4,300 + 사실 전제 211(식품 78 · 건기식 68 · 화장품 65) · 로컬 대조 10-14
if GOLDEN_SHA in DEV_EXPECT and not SMOKE and USE_WORD_ROWS is False:
    _got = {k: TRAIN_STATS[k] for k in EXPECT_TRAIN}
    if _got != EXPECT_TRAIN:
        raise SystemExit(f"🔴 학습 행 수가 기대와 다르다 — 받은 {_got} · 기대 {EXPECT_TRAIN}. 학습셋 코드를 건드렸는지 본다")
    print("✅ 학습셋이 D4 기록에 적은 수와 같다")
for _ in [0]:
    assert not (set(r["id"] for r in TRAIN_ROWS) & (set(r["id"] for r in dev_rows) | set(r["id"] for r in SYN_EVAL))), "학습 행에 dev · 측정 행이 섞였다"
    assert not ({norm_text(r["text"]) for r in TRAIN_ROWS} & dev_texts), "학습 행에 dev 와 같은 문구가 있다"
    assert all(r["split"] == "train" or r["origin"] == "synthetic_lawful" for r in TRAIN_ROWS), "골든 train 도 정상 합성도 아닌 행이 학습에 있다"

## 5. 임베딩 — OpenAI API (한 번 부르고 Drive 에 저장)

입력 글 — (가) `품목: <식품|건기식|화장품|모름>\n문장: <문장>` · (나) 품목 자리에 전부 `모름`. 같은 글은 한 번만 보낸다.


In [ ]:
import getpass
from openai import OpenAI
try:
    from google.colab import userdata
    _key = userdata.get("OPENAI_API_KEY")
except Exception:
    _key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OPENAI_API_KEY: ")
client = OpenAI(api_key=_key)
del _key

def emb_text(cat, text):
    return f"품목: {cat}\n문장: {text}"

def texts_for(rows, mode):
    return [emb_text(cat_of(r) if mode == "품목" else CAT_UNKNOWN, r["text"]) for r in rows]

NEED = set()
for rs in (TRAIN_ROWS, dev_rows, SYN_EVAL):
    for m in MODES:
        NEED |= set(texts_for(rs, m))
NEED = sorted(NEED)
print(f"보낼 글 {len(NEED):,}개 · 글자 {sum(map(len, NEED)):,}")

def embed_all(model, texts, path, bs=256):
    cache = {}
    if os.path.exists(path):
        z = np.load(path, allow_pickle=True)
        cache = dict(zip(z["texts"].tolist(), z["vecs"]))
    todo = [t for t in texts if t not in cache]
    usage = 0
    for i in range(0, len(todo), bs):
        part = todo[i:i + bs]
        for attempt in range(6):
            try:
                resp = client.embeddings.create(model=model, input=part)
                break
            except Exception as e:
                wait = 2 ** attempt * 5
                print(f"  ⏳ {type(e).__name__} — {wait}초 뒤 다시 ({attempt + 1}/6)")
                time.sleep(wait)
        else:
            raise SystemExit("🔴 임베딩 요청이 계속 실패한다")
        usage += resp.usage.total_tokens
        for t, d in zip(part, resp.data):
            cache[t] = np.asarray(d.embedding, dtype=np.float32)
        if (i // bs) % 10 == 0:
            print(f"  {model}: {min(i + bs, len(todo)):,}/{len(todo):,}")
    ks = list(cache)
    np.savez(path, texts=np.array(ks, dtype=object), vecs=np.stack([cache[k] for k in ks]))
    return cache, usage

EMB, USAGE = {}, {}
for name, model in EMB_MODELS.items():
    t0 = time.time()
    EMB[name], USAGE[name] = embed_all(model, NEED, f"{OUT_ROOT}/emb_{name}.npz")
    dim = len(next(iter(EMB[name].values())))
    print(f"✅ {name} ({model}) · 차원 {dim} · 이번에 쓴 토큰 {USAGE[name]:,} · {time.time() - t0:.0f}초")


## 6. 분류기 — 유형마다 로지스틱 회귀 (고정값)

- 학습 행: 품목이 있는 행은 (가) 품목판 · (나) 「모름」판을 가중치 0.5 씩 · 품목이 없는 행은 「모름」판 1.0 — D4-10 의 50% 「모름」 섞기와 같은 기대값
- 표준화(학습 행으로 맞춤) → `LogisticRegression(C=1.0, class_weight="balanced")` · 유형마다 따로 · 양성이 없는 칸은 0
- 확률을 dev 749행 · 정상 문구 측정 행에 두 입력으로 적는다 — 문턱은 로컬 스크립트가 D-324 로 고른다


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

def multihot(labels):
    v = np.zeros(N_ALL, dtype=np.float32)
    for l in labels:
        v[LABEL2ID[l]] = 1.0
    return v

def X_of(name, rows, mode):
    return np.stack([EMB[name][t] for t in texts_for(rows, mode)])

def macro_prauc(y, p, n=N_SEL):
    v = [float(average_precision_score(y[:, i], p[:, i])) for i in range(n) if 0 < y[:, i].sum() < len(y)]
    return float(np.mean(v)) if v else float("nan")

Y_DEV = np.stack([multihot(r["target"]) for r in dev_rows]).astype(int)
RESULTS = {}
for name in EMB_MODELS:
    t0 = time.time()
    has_cat = np.array([cat_of(r) != CAT_UNKNOWN for r in TRAIN_ROWS])
    Xa, Xu = X_of(name, TRAIN_ROWS, "품목"), X_of(name, TRAIN_ROWS, "모름")
    Ytr = np.stack([multihot(r["target"]) for r in TRAIN_ROWS])
    X = np.concatenate([Xa[has_cat], Xu])
    Y = np.concatenate([Ytr[has_cat], Ytr])
    W = np.concatenate([np.full(has_cat.sum(), 0.5), np.where(has_cat, 0.5, 1.0)])
    sc = StandardScaler().fit(X, sample_weight=W)
    Xs = sc.transform(X)
    heads = []
    for i, l in enumerate(LABEL_LIST):
        if Y[:, i].sum() == 0:
            heads.append(None); continue
        h = LogisticRegression(C=LR_C, class_weight=LR_CLASS_WEIGHT, max_iter=LR_MAX_ITER)
        h.fit(Xs, Y[:, i], sample_weight=W)
        heads.append(h)
    def predict(rows, mode):
        Z = sc.transform(X_of(name, rows, mode))
        return np.stack([h.predict_proba(Z)[:, 1] if h is not None else np.zeros(len(rows)) for h in heads], 1)
    R = {"minutes": (time.time() - t0) / 60}
    for m in MODES:
        R[f"dev_p_{m}"] = predict(dev_rows, m)
        R[f"syn_p_{m}"] = predict(SYN_EVAL, m)
        R[f"prauc6_{m}"] = macro_prauc(Y_DEV[SEL_MASK], R[f"dev_p_{m}"][SEL_MASK])
    RESULTS[name] = R
    print(f"  {name}: 6종 dev macro PR-AUC (가) 품목 {R['prauc6_품목']:.4f} · (나) 모름 {R['prauc6_모름']:.4f} · 학습 {R['minutes']:.1f}분 · 학습 표본 {len(X):,}")
print("D4-10 (참고 · KcBERT 파인튜닝 · 시드 평균) — (가) 0.7399 · (나) 0.7177")


## 7. 응답 시간 — 문장 하나를 API 로 임베딩하는 데 걸리는 시간 (참고)

dev 문장 30개를 하나씩(배치 1) 보낸다. 파인튜닝 인코더는 CPU 배치 1 중앙값 42 ms (클라우드 2코어 · D4-10).


In [ ]:
LAT = {}
_sample = texts_for(dev_rows[:30], "품목")
for name, model in EMB_MODELS.items():
    ms = []
    for t in _sample:
        t0 = time.perf_counter()
        client.embeddings.create(model=model, input=[t])
        ms.append((time.perf_counter() - t0) * 1000)
    LAT[name] = {"median_ms": float(np.median(ms)), "p90_ms": float(np.percentile(ms, 90)), "n": len(ms)}
    print(f"  {name}: 중앙값 {LAT[name]['median_ms']:.0f} ms · 90% {LAT[name]['p90_ms']:.0f} ms (Colab → OpenAI · 네트워크 포함)")


## 8. 저장 — 채점 스크립트와 같은 형식의 CSV · 묶음

결정적 학습이라 같은 확률을 시드 48 · 49 · 50 이름으로 세 번 적는다(채점 스크립트가 시드 셋을 읽는다 · 표준편차는 0 으로 나온다).


In [ ]:
out = []
for name, R in RESULTS.items():
    tag = run_tag(name)
    for sd in SEEDS:
        for m in MODES:
            p = f"{OUT_ROOT}/{tag}_{m}_seed{sd}_dev행별확률.csv"
            with open(p, "w", encoding="utf-8-sig", newline="") as f:
                w = csv.writer(f)
                w.writerow(["id", "조건", "bucket", "이유구역", "품목입력"] + [f"p_{l}" for l in LABEL_LIST])
                for r, row in zip(dev_rows, R[f"dev_p_{m}"]):
                    w.writerow([r["id"], r["cond"] or "", r["bucket"], int(is_reason(r)), cat_of(r) if m == "품목" else CAT_UNKNOWN] + [f"{x:.6f}" for x in row])
            out.append(os.path.basename(p))
            p = f"{OUT_ROOT}/{tag}_{m}_seed{sd}_합성정상_행별확률.csv"
            with open(p, "w", encoding="utf-8-sig", newline="") as f:
                w = csv.writer(f)
                w.writerow(["id", "품목", "갈래"] + [f"p_{l}" for l in LABEL_LIST])
                for r, row in zip(SYN_EVAL, R[f"syn_p_{m}"]):
                    w.writerow([r["id"], r["cat"], ("애매" if r["amb_mark"] else f"참고-{r['review']}") if r["amb"] else "합성정상dev"] + [f"{x:.6f}" for x in row])
            out.append(os.path.basename(p))
SUMMARY = {"notebook": NOTEBOOK, "golden_sha256": GOLDEN_SHA, "dev_sha": DEV_SHA, "dev_rows": len(dev_rows), "syn_sha256": SYN_SHA,
           "train": TRAIN_STATS, "emb_models": EMB_MODELS, "classifier": {"C": LR_C, "class_weight": LR_CLASS_WEIGHT, "scaler": "StandardScaler"},
           "tokens_used": USAGE, "latency": LAT, "prauc6": {n: {m: R[f"prauc6_{m}"] for m in MODES} for n, R in RESULTS.items()}}
json.dump(SUMMARY, open(f"{OUT_ROOT}/results_E1-02.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)
with zipfile.ZipFile(f"{OUT_ROOT}/E1-02_csv.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in out:
        z.write(f"{OUT_ROOT}/{f}", f)
    z.write(f"{OUT_ROOT}/results_E1-02.json", "results_E1-02.json")
print(f"📦 받을 파일: {OUT_ROOT}/E1-02_csv.zip ({len(out)}개 + results) — PC 의 Downloads\\copylane_D4\\E1-02_csv\\ 에 풀어 둔다")


## 9. 다음 단계

```
uv run python docs\psj\e2e_prototype\e1_00_electra.py --run E1-02s --dir <D4-00_csv> --d10-dir <D4-10_csv> --e1-dir <E1-02_csv> --review-csv <검수판> --out-md docs\psj\reports\E1-02_OpenAI임베딩_small_dev_YYYYMMDD.md
uv run python docs\psj\e2e_prototype\e1_00_electra.py --run E1-02l --dir <D4-00_csv> --d10-dir <D4-10_csv> --e1-dir <E1-02_csv> --review-csv <검수판> --out-md docs\psj\reports\E1-02_OpenAI임베딩_large_dev_YYYYMMDD.md
```

올리지 않는 것 — 임베딩 npz · 행별 확률 CSV · 실행 출력 · `golden.jsonl` · 정상 문구 CSV · **API 키** (D-249).
